In [ ]:
# Mount Google Drive (if using Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False
    print("Not running in Colab")

Mounted at /content/drive


# [GPT Link](https://chatgpt.com/share/69b327a6-da94-8005-80c3-f7f09081bc40)

## [Word File](https://docs.google.com/document/d/1ufwWzABa8MhAhmbVzq7El8mLDy3Ra8m-l1jWfWW9ZRE/edit?usp=sharing)

# [Policy_en CSV Link](https://drive.google.com/file/d/17ShQTtD0Ryy6wycrrLfkJ3Ak-80nkp-v/view?usp=sharing)

In [ ]:
import pandas as pd
Policy_Csv = "/content/drive/MyDrive/hf_models/agent_project/Policy_en.csv"
df = pd.read_csv(Policy_Csv)
df

,policy_id,section,policy_name,rule_id,rule_description,threshold_value,condition,approval_required,risk_level
0,1,Financial,Hardware Purchase Policy,1,Purchases up to $1500 are auto-approved,1500.0,amount <= 1500,No,Medium
1,1,Financial,Hardware Purchase Policy,2,Purchases between $1501–$3000 require manager ...,3000.0,1500 < amount <= 3000,Manager,Medium
2,1,Financial,Hardware Purchase Policy,3,"Purchases above $3000 require manager, finance...",3000.0,amount > 3000,Manager+Finance+CTO,Medium
3,1,Financial,Hardware Purchase Policy,4,Annual hardware budget per employee is $2500,2500.0,annual_budget_limit,No,Medium
4,1,Financial,Hardware Purchase Policy,5,Vendor must be selected from approved vendor list,NaN,vendor_approved == true,No,Medium
5,2,Financial,Travel Expense Policy,1,Economy class required for flights under 6 hours,6.0,flight_hours <= 6,No,Medium
6,2,Financial,Travel Expense Policy,2,Business class allowed for flights over 6 hour...,6.0,flight_hours > 6 AND role == executive,Manager,Medium
7,2,Financial,Travel Expense Policy,3,Hotel cost must not exceed $250 regional,250.0,region == regional,No,Medium
8,2,Financial,Travel Expense Policy,4,Hotel cost must not exceed $400 international ...,400.0,region == international,No,Medium
9,2,Financial,Travel Expense Policy,5,Receipts mandatory for reimbursement,NaN,receipt_provided == true,No,Medium


In [ ]:
df= df.drop(columns=["policy_id","rule_id"])
df["full_text"] = df.astype(str).agg(" | ".join, axis=1)
df.head()

,section,policy_name,rule_description,threshold_value,condition,approval_required,risk_level,full_text
0,Financial,Hardware Purchase Policy,Purchases up to $1500 are auto-approved,1500.0,amount <= 1500,No,Medium,Financial | Hardware Purchase Policy | Purchas...
1,Financial,Hardware Purchase Policy,Purchases between $1501–$3000 require manager ...,3000.0,1500 < amount <= 3000,Manager,Medium,Financial | Hardware Purchase Policy | Purchas...
2,Financial,Hardware Purchase Policy,"Purchases above $3000 require manager, finance...",3000.0,amount > 3000,Manager+Finance+CTO,Medium,Financial | Hardware Purchase Policy | Purchas...
3,Financial,Hardware Purchase Policy,Annual hardware budget per employee is $2500,2500.0,annual_budget_limit,No,Medium,Financial | Hardware Purchase Policy | Annual ...
4,Financial,Hardware Purchase Policy,Vendor must be selected from approved vendor list,NaN,vendor_approved == true,No,Medium,Financial | Hardware Purchase Policy | Vendor ...


In [ ]:
df["full_text"][0]

'Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium'

In [ ]:
# Target
#1) Matching row with userQ  ( Semantic Search)
#2) Matching rule with condition col. (keyword Search)

In [ ]:
#  Semantic Search

# 1) Embedding


from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")
policy_texts = df["full_text"].tolist()

policy_embeddings = embedder.encode(
    policy_texts,
    normalize_embeddings=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
policy_texts[0]

'Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium'

In [ ]:
policy_embeddings[0]

array([-1.37061914e-02,  4.47236598e-02, -1.61637012e-02, -5.36067747e-02,
       -1.91182469e-03,  2.49939617e-02,  2.33572293e-02,  4.44731191e-02,
       -4.84552272e-02,  2.69257501e-02,  6.35468215e-02, -1.86100081e-02,
        6.56950194e-03, -5.27026579e-02, -8.51291865e-02, -1.72419809e-02,
        5.09684980e-02, -1.25831291e-01, -2.87466925e-02,  4.32613445e-03,
        2.74031181e-02, -3.38292122e-02, -1.66924689e-02,  1.64940860e-02,
        3.45937572e-02,  6.16535405e-03,  5.32277450e-02,  2.17648651e-02,
       -7.00197648e-03,  3.58746126e-02, -1.00474589e-01, -3.03701963e-02,
        9.29182172e-02, -5.24217673e-02,  6.00386821e-02, -7.98505619e-02,
       -6.13256879e-02, -1.00962795e-01, -3.90885957e-02, -1.18482850e-01,
        1.34699391e-02, -8.85257944e-02, -8.76408815e-02,  3.28780189e-02,
        6.45360574e-02,  2.00531203e-02, -6.23312108e-02,  6.48792461e-02,
        2.42649857e-03,  8.24130699e-02,  4.28119069e-03,  7.37885088e-02,
        4.21453267e-02, -

In [ ]:
userQ =  "i need left form my job in 3 Days"
userQ = "I need buy device with 4500$"

userQ_embedding = embedder.encode(
    [userQ],
    normalize_embeddings=True
)

In [ ]:
# userQ_embedding  and policy_embeddings

In [ ]:
score = (  userQ_embedding @ policy_embeddings.T).squeeze()

# Cosine Similarity apply with normalize_embeddings=True in embedder.encode()
# score = (userQ_embedding @ policy_embeddings.T).squeeze() / (
  #  np.linalg.norm(userQ_embedding) *
 #   np.linalg.norm(policy_embeddings, axis=1)
# )

top_k = 5
top_idx = np.argsort(-score)[:top_k]

for i in top_idx:
    print("Score:", round(score[i], 3))
    print("Text :", df.iloc[i]["full_text"])
    print("-"*60)

Score: 0.478
Text : Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Manager | Medium
------------------------------------------------------------
Score: 0.369
Text : Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium
------------------------------------------------------------
Score: 0.357
Text : Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Finance+CTO | Medium
------------------------------------------------------------
Score: 0.311
Text : Financial | Hardware Purchase Policy | Annual hardware budget per employee is $2500 | 2500.0 | annual_budget_limit | No | Medium
------------------------------------------------------------
Score: 0.263
Text : Financial | Hardware Purchase Policy | Vendor must be selected from approved vendor list 

In [ ]:
def retrive_cand(userQ,policy_embeddings):
  out = []
  userQ_embedding = embedder.encode(
    [userQ],
    normalize_embeddings=True
  )
  score = (  userQ_embedding @ policy_embeddings.T).squeeze()
  top_k = 5
  top_idx = np.argsort(-score)[:top_k]
  for i in top_idx:
    #  print("Score:", round(score[i], 3))

    out.append({ "Score":round(score[i], 3)  ,  "full_text":df.iloc[i]["full_text"]})
     # print("Text :", df.iloc[i]["full_text"])
    #  print("-"*60)
  return out

In [ ]:
userQ =  "i need left form my job in 3 Days"
retrive_cand(userQ,policy_embeddings)

[{'Score': np.float32(0.241),
  'full_text': 'Legal | Contract Termination Policy | Contracts must include minimum 30-day termination notice | 30.0 | termination_notice_days >= 30 | Legal | High'},
 {'Score': np.float32(0.239),
  'full_text': 'HR | Remote Work Equipment Policy | Replacement allowed every 3 years | 3.0 | years_since_last_replacement >= 3 | Manager | Medium'},
 {'Score': np.float32(0.161),
  'full_text': 'Legal | Contract Termination Policy | Indemnification clauses must be reviewed by legal | nan | indemnification_clause == true | Legal | High'},
 {'Score': np.float32(0.158),
  'full_text': 'Legal | Vendor Agreement Policy | Payment terms must not exceed Net 60 unless CFO approval | 60.0 | payment_terms_days <= 60 | CFO_if_exceeds | High'},
 {'Score': np.float32(0.152),
  'full_text': 'Legal | Contract Termination Policy | Liability cap must not exceed 12 months of contract value | 12.0 | liability_cap_months <= 12 | Legal | High'}]

In [ ]:
userQ = "I need buy device with 4500$"
top5Cand = retrive_cand(userQ,policy_embeddings)
top5Cand

[{'Score': np.float32(0.478),
  'full_text': 'Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Manager | Medium'},
 {'Score': np.float32(0.369),
  'full_text': 'Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium'},
 {'Score': np.float32(0.357),
  'full_text': 'Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Finance+CTO | Medium'},
 {'Score': np.float32(0.311),
  'full_text': 'Financial | Hardware Purchase Policy | Annual hardware budget per employee is $2500 | 2500.0 | annual_budget_limit | No | Medium'},
 {'Score': np.float32(0.263),
  'full_text': 'Financial | Hardware Purchase Policy | Vendor must be selected from approved vendor list | nan | vendor_approved == true | No | Medium'}]

In [ ]:
# 2) Matching rule with condition col. (keyword Search)

# A) extract number form userQ
# B) extract cond from full text in top5 cand
# C) check cond ???

In [ ]:
import re

def extract_amount(text):
    match = re.search(r'[\d,]+', text)
    if match:
        return int(match.group().replace(",", ""))
    return None

amount_userQ = extract_amount(userQ)
amount_userQ

4500

In [ ]:
def fullsystem(userQ,policy_embeddings):
  out = []
  amount_userQ = extract_amount(userQ) # 1

  userQ_embedding = embedder.encode(
    [userQ],
    normalize_embeddings=True
  )
  score = (  userQ_embedding @ policy_embeddings.T).squeeze()
  top_k = 5
  top_idx = np.argsort(-score)[:top_k]
  num = 0
  matched = []
  for i in top_idx:
   # print(f"----------- top {num+1} -----------")
    condition = df.iloc[i]["condition"]
    #print(condition)
    expr= condition.replace("amount",str(amount_userQ))
    #print(expr)
    try:
      if eval(expr) == True:
        matched.append({ "Score":round(score[i], 3)  ,  "full_text":df.iloc[i]["full_text"]})

    except:
      pass
    num +=1

  return matched

In [ ]:
userQ = "I need buy device with 4500$"

fullsystem(userQ,policy_embeddings)

[{'Score': np.float32(0.357),
  'full_text': 'Financial | Hardware Purchase Policy | Purchases above $3000 require manager, finance, and CTO approval | 3000.0 | amount > 3000 | Manager+Finance+CTO | Medium'}]

In [ ]:
userQ = "I need buy device with 1000$"

fullsystem(userQ,policy_embeddings)

[{'Score': np.float32(0.34),
  'full_text': 'Financial | Hardware Purchase Policy | Purchases up to $1500 are auto-approved | 1500.0 | amount <= 1500 | No | Medium'}]

In [ ]:
userQ = "I need buy device with 2500$"

fullsystem(userQ,policy_embeddings)

[{'Score': np.float32(0.437),
  'full_text': 'Financial | Hardware Purchase Policy | Purchases between $1501–$3000 require manager approval and 2 quotes | 3000.0 | 1500 < amount <= 3000 | Manager | Medium'}]

In [ ]:
# Task 1
# top5 to LLM >> and get top 1 matching to userQ


# Task 2:
# Handle flight_hours prog. & amount